In [1]:
# VERBA — Multilingual Translation Prototype

### Connecting Languages, Connecting People

"VERBA is an academic multilingual translation prototype..."

'VERBA is an academic multilingual translation prototype...'

In [2]:

# Install the libraries required by the translation model

%pip install -q "transformers<5" sentencepiece accelerate gradio

print("Dependencies installed successfully.")


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
Dependencies installed successfully.


In [4]:

from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


In [5]:

# Import required libraries

import os
import sys
import time
import csv
import statistics

import torch
import transformers
import sentencepiece
import accelerate
import gradio as gr

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Select available hardware
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Display environment information
print("VERBA Environment")
print("-" * 35)
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("SentencePiece:", sentencepiece.__version__)
print("Accelerate:", accelerate.__version__)
print("Gradio:", gr.__version__)
print("Device:", DEVICE)

if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )
else:
    print("GPU not detected. Running on CPU.")


VERBA Environment
-----------------------------------
Python: 3.13.16
PyTorch: 2.14.1+cu130
Transformers: 4.57.6
SentencePiece: 0.2.2
Accelerate: 1.15.0
Gradio: 6.17.3
Device: cuda
GPU: Tesla T4
VRAM: 14.56 GB


In [6]:
# Load the pretrained NLLB-200 translation model

MODEL_NAME = "facebook/nllb-200-distilled-600M"
MODEL_PATH = "/content/drive/MyDrive/VERBA/models/nllb-200-distilled-600M"

MODEL_DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print("Loading VERBA translation model from Hugging Face...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=MODEL_DTYPE
)

model = model.to(DEVICE)
model.eval()

# Save the complete checkpoint to Google Drive
model.save_pretrained(MODEL_PATH, safe_serialization=True)
tokenizer.save_pretrained(MODEL_PATH)

print("\nVERBA model loaded and saved successfully!")
print("Parameters:", round(sum(p.numel() for p in model.parameters()) / 1_000_000, 2), "million")
print("Device:", DEVICE)
print("Precision:", MODEL_DTYPE)
print("Checkpoint:", MODEL_PATH)

Loading VERBA translation model from Hugging Face...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

... (3 outputs left out)

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(



VERBA model loaded and saved successfully!
Parameters: 615.07 million
Device: cuda
Precision: torch.float16
Checkpoint: /content/drive/MyDrive/VERBA/models/nllb-200-distilled-600M


In [8]:
# Configure supported languages


# VERBA — Expanded Language Configuration

LANGUAGES = {
    # Indian Languages
    "English": "eng_Latn",
    "Hindi": "hin_Deva",
    "Marathi": "mar_Deva",
    "Gujarati": "guj_Gujr",
    "Bengali": "ben_Beng",
    "Tamil": "tam_Taml",
    "Telugu": "tel_Telu",
    "Punjabi": "pan_Guru",
    "Kannada": "kan_Knda",
    "Malayalam": "mal_Mlym",
    "Urdu": "urd_Arab",

    # Asian Languages
    "Japanese": "jpn_Jpan",
    "Chinese": "zho_Hans",
    "Korean": "kor_Hang",
    "Thai": "tha_Thai",
    "Indonesian": "ind_Latn",

    # European Languages
    "Spanish": "spa_Latn",
    "French": "fra_Latn",
    "German": "deu_Latn",
    "Italian": "ita_Latn",
    "Portuguese": "por_Latn",
    "Russian": "rus_Cyrl"
}

print("VERBA supports", len(LANGUAGES), "languages.")



# VERBA — Updated Translation Function

def translate(text, source_language, target_language):

    if not text or not text.strip():
        raise ValueError("Please enter some text.")

    if source_language not in LANGUAGES:
        raise ValueError("Invalid source language.")

    if target_language not in LANGUAGES:
        raise ValueError("Invalid target language.")

    if source_language == target_language:
        return text

    tokenizer.src_lang = LANGUAGES[source_language]

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(DEVICE)

    target_id = tokenizer.convert_tokens_to_ids(
        LANGUAGES[target_language]
    )

    if target_id == tokenizer.unk_token_id:
        raise ValueError("Unsupported language code.")

    with torch.no_grad():
        output = model.generate(
            **inputs,
            forced_bos_token_id=target_id,
            max_new_tokens=128,
            num_beams=4
        )

    translated_text = tokenizer.batch_decode(
        output,
        skip_special_tokens=True
    )[0]

    return translated_text


print("VERBA Translation Engine Ready!")



# VERBA — Language Verification

print("VERBA Language Verification")
print("-" * 40)

for language, code in LANGUAGES.items():

    token_id = tokenizer.convert_tokens_to_ids(code)

    if token_id == tokenizer.unk_token_id:
        print(language, "❌ Not supported")
    else:
        print(language, "✅ Available")

print("\nLanguage verification completed!")



VERBA supports 22 languages.
VERBA Translation Engine Ready!
VERBA Language Verification
----------------------------------------
English ✅ Available
Hindi ✅ Available
Marathi ✅ Available
Gujarati ✅ Available
Bengali ✅ Available
Tamil ✅ Available
Telugu ✅ Available
Punjabi ✅ Available
Kannada ✅ Available
Malayalam ✅ Available
Urdu ✅ Available
Japanese ✅ Available
Chinese ✅ Available
Korean ✅ Available
Thai ✅ Available
Indonesian ✅ Available
Spanish ✅ Available
French ✅ Available
German ✅ Available
Italian ✅ Available
Portuguese ✅ Available
Russian ✅ Available

Language verification completed!


In [9]:

# VERBA — Multilingual Translation Demonstration

sample_text = "Welcome to Palghar. We are building a better future together."

target_languages = [
    "Hindi",
    "Marathi",
    "Gujarati",
    "Bengali",
    "Tamil",
    "Telugu",
    "Spanish",
    "French",
    "Japanese",
    "German"
]

print("VERBA Translation Demonstration")
print("-" * 50)
print("Source English:", sample_text)

for language in target_languages:

    result = translate(
        sample_text,
        "English",
        language
    )

    print(f"\n{language}:")
    print(result)


VERBA Translation Demonstration
--------------------------------------------------
Source English: Welcome to Palghar. We are building a better future together.

Hindi:
पालघर में आपका स्वागत है, हम एक साथ एक बेहतर भविष्य का निर्माण कर रहे हैं।

Marathi:
पालघरमध्ये आपले स्वागत आहे. आम्ही एकत्रितपणे एक चांगले भविष्य घडवत आहोत.

Gujarati:
પાલ્ઘરમાં આપનું સ્વાગત છે. અમે સાથે મળીને વધુ સારા ભવિષ્યનું નિર્માણ કરી રહ્યા છીએ.

Bengali:
পলগারে স্বাগতম, আমরা একসাথে আরও ভালো ভবিষ্যৎ গড়ে তুলছি।

Tamil:
பால்காரில் வரவேற்கிறோம். நாம் ஒரு சிறந்த எதிர்காலத்தை ஒன்றாக உருவாக்குகிறோம்.

Telugu:
పల్ఘర్ కు స్వాగతం. మనం కలిసి మెరుగైన భవిష్యత్తును నిర్మిస్తున్నాం.

Spanish:
Bienvenidos a Palghar, estamos construyendo un futuro mejor juntos.

French:
Bienvenue à Palghar, nous construisons ensemble un avenir meilleur.

Japanese:
パルガーへようこそ 一緒により良い未来を築こう

German:
Willkommen in Palghar, wir bauen eine bessere Zukunft zusammen.


In [ ]:
# VERBA — Translation Performance Benchmark

evaluation_sentences = [
    "Welcome to Palghar.",
    "How are you today?",
    "Please bring your identity card.",
    "The hospital is located near the main road.",
    "We are building a better future together.",
    "Clean water is important for every community.",
    "The bus will arrive in ten minutes.",
    "Education creates new opportunities.",
    "Please contact us for more information.",
    "Thank you for your cooperation."
]

target_languages = ["Hindi", "Marathi", "Spanish", "Japanese"]

benchmark_results = []

# Warm-up run (not included in measurements)
translate("Welcome to Palghar.", "English", "Hindi")

if DEVICE == "cuda":
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

print("Running VERBA performance benchmark...")
print("Total translations:", len(evaluation_sentences) * len(target_languages))

for sentence in evaluation_sentences:
    for language in target_languages:

        if DEVICE == "cuda":
            torch.cuda.synchronize()

        start_time = time.perf_counter()

        translated_text = translate(
            sentence,
            "English",
            language
        )

        if DEVICE == "cuda":
            torch.cuda.synchronize()

        latency = time.perf_counter() - start_time

        benchmark_results.append({
            "source_text": sentence,
            "target_language": language,
            "translated_text": translated_text,
            "latency_seconds": round(latency, 4)
        })

# Calculate performance statistics
latencies = [
    result["latency_seconds"]
    for result in benchmark_results
]

# Save benchmark results
report_path = "/content/drive/MyDrive/VERBA/reports"

os.makedirs(report_path, exist_ok=True)

csv_path = os.path.join(
    report_path,
    "performance_benchmark.csv"
)

with open(csv_path, "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=benchmark_results[0].keys()
    )
    writer.writeheader()
    writer.writerows(benchmark_results)

# Display summary
print("\nVERBA Performance Summary")
print("-" * 40)
print("Total translations:", len(latencies))
print("Average latency:", round(statistics.mean(latencies), 3), "seconds")
print("Median latency:", round(statistics.median(latencies), 3), "seconds")
print("Fastest translation:", round(min(latencies), 3), "seconds")
print("Slowest translation:", round(max(latencies), 3), "seconds")

for language in target_languages:
    language_times = [
        result["latency_seconds"]
        for result in benchmark_results
        if result["target_language"] == language
    ]

    print(
        f"{language}:",
        round(statistics.mean(language_times), 3),
        "seconds average"
    )

if DEVICE == "cuda":
    peak_memory = torch.cuda.max_memory_allocated() / (1024 ** 3)
    print("Peak GPU memory:", round(peak_memory, 3), "GB")

print("\nBenchmark report saved:", csv_path)

In [12]:

# ============================================
# VERBA 2.0 — MODERN MULTILINGUAL TRANSLATOR
# ============================================

import gradio as gr

# Translation function
def run_translation(text, source_language, target_language):

    if not text or not text.strip():
        return "Please enter text to translate."

    try:
        return translate(text, source_language, target_language)

    except Exception as error:
        return f"Translation Error: {str(error)}"


# Swap languages
def swap_languages(source, target):
    return target, source


# Clear text fields
def clear_fields():
    return "", ""


# Modern interface styling
custom_css = """
.gradio-container {
    background: linear-gradient(135deg, #10172e, #20204a, #301b52);
}

#main-title {
    text-align: center;
    padding: 30px;
    border-radius: 18px;
    background: linear-gradient(90deg, #292e63, #663399);
}

#main-title h1 {
    font-size: 45px;
    color: white;
}

#main-title p {
    color: #e1dcff;
}

.gr-button-primary {
    background: linear-gradient(90deg, #6255e7, #9b59d0) !important;
    border: none !important;
}
"""


# Create VERBA application
with gr.Blocks(title="VERBA - AI Translator") as demo:

    # Header
    gr.HTML("""
    <div id="main-title">
        <h1>🌐 VERBA</h1>
        <p>Connecting Languages, Connecting People</p>
        <p>AI-Powered Multilingual Translation Platform</p>
    </div>
    """)

    gr.Markdown(
        "## ✨ Translate your thoughts into another language"
    )

    # Language selection
    with gr.Row():

        source_language = gr.Dropdown(
            choices=list(LANGUAGES.keys()),
            value="English",
            label="🌍 Translate From"
        )

        swap_button = gr.Button("⇄ Swap Languages")

        target_language = gr.Dropdown(
            choices=list(LANGUAGES.keys()),
            value="Marathi",
            label="🎯 Translate To"
        )

    # Input and output boxes
    with gr.Row():

        input_text = gr.Textbox(
            label="📝 Enter Your Text",
            placeholder="Type something here...",
            lines=8
        )

        output_text = gr.Textbox(
            label="🌐 Translation Result",
            placeholder="Your translation appears here...",
            lines=8,
            interactive=False
        )

    # Action buttons
    with gr.Row():

        translate_button = gr.Button(
            "✨ Translate Now",
            variant="primary"
        )

        clear_button = gr.Button("🗑 Clear")

    # Example sentences
    gr.Markdown("## 💡 Try These Examples")

    gr.Examples(
        examples=[
            ["Hello, how are you?", "English", "Hindi"],
            ["Welcome to Palghar.", "English", "Marathi"],
            ["Education is the key to success.", "English", "Gujarati"],
            ["Thank you for your help.", "English", "Spanish"],
            ["Good morning.", "English", "Japanese"],
            ["Technology connects the world.", "English", "French"],
            ["Knowledge is power.", "English", "German"]
        ],
        inputs=[
            input_text,
            source_language,
            target_language
        ]
    )

    # About section
    gr.Markdown("""
    ---

    ## 🚀 About VERBA

    VERBA uses Meta AI's pretrained
    **NLLB-200 Distilled 600M** translation model.

    ### Features

    - 22 language options
    - AI-powered text translation
    - Language swapping
    - Easy-to-use interface
    - Multiple language support

    ---

    ⚠️ Academic translation prototype.
    Please verify translations before important use.

    <center>
    Built with Python, PyTorch and Gradio
    | VERBA 2.0
    </center>
    """)

    # Connect buttons
    translate_button.click(
        fn=run_translation,
        inputs=[
            input_text,
            source_language,
            target_language
        ],
        outputs=output_text
    )

    swap_button.click(
        fn=swap_languages,
        inputs=[
            source_language,
            target_language
        ],
        outputs=[
            source_language,
            target_language
        ]
    )

    clear_button.click(
        fn=clear_fields,
        inputs=[],
        outputs=[
            input_text,
            output_text
        ]
    )


# Launch VERBA with custom theme and CSS
demo.launch(
    share=True,
    theme=gr.themes.Soft(),
    css=custom_css
)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://07c7636a048794610d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
